In [12]:
import sys, os
import numpy as np
import pandas as pd
from scipy.stats import entropy
from collections import defaultdict

from csc import *
from exp_utils import *

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [22]:
import warnings
warnings.filterwarnings('ignore') 

In [38]:
potato_duplicate_questions = [14, 83, 121]
method = "nli-batch"

models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-v0.3-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0",
    "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}
pm_symbol = u"\u00B1"

In [39]:
p = "preprompt"

In [40]:
for model in models:
    uncertainty_df = pd.read_csv(f"data/{p}/{model}/uncertainty.csv")
    for dataset in datasets:
        row = ""
        if dataset == "hotpot_qa_final":
            row += "\\multirow{4}{*}{%s}"%(model_rename[model])

        fname = f"data/{p}/{model}/{dataset}_results.json"
        
        try:
            with open(fname) as f:
                summary = json.load(f)
        except Exception as e:
            print("\tModel-dataset pair not found.")
            continue

        # only iterate over the ids in the spreadsheet
        question_ids = uncertainty_df[uncertainty_df["dataset"]==dataset]["id"].unique()
        kl_values = []
        prob_diff_values = []
        tvd_values = []
        for question_id in question_ids:
            log_probs = summary[str(question_id)]["log_probs"]
            cluster_ids = summary[str(question_id)]["cluster_ids"][method]["100"]
            
            cluster_counts = np.bincount(cluster_ids)
            probs = cluster_counts / len(cluster_ids)
            empirical_probs = np.array(probs)

            unique_clusters = np.unique(cluster_ids)
            cluster_probs = defaultdict(float)
            probabilities = np.exp(log_probs)
            for cluster, prob in zip(cluster_ids, probabilities):
                cluster_probs[cluster] += prob
            total_prob = sum(cluster_probs.values())
            probs = np.array(
                [cluster_probs[cluster] / total_prob for cluster in unique_clusters]
            )
            rbmci_probs = np.array(probs)

            # metrics
            kl = entropy(empirical_probs, rbmci_probs)
            elementwise_diff = np.absolute(rbmci_probs-empirical_probs)
            tvd = 0.5 * elementwise_diff.sum() # total variation distance
            per_class_prob_diff = elementwise_diff.mean() # this is just MAE
            
            kl_values.append(kl)
            tvd_values.append(tvd)
            prob_diff_values.append(per_class_prob_diff)
        
        kl_mean = np.mean(kl_values)
        tvd_mean = np.mean(tvd_values)
        prob_diff_mean = np.mean(prob_diff_values)

        row += f"  &  {datasets[dataset]}  &  {kl_mean:.3f}  &  {tvd_mean:.3f}  &  {prob_diff_mean:.3f} \\\\"
        print(row)
    print("\\hline")

\multirow{4}{*}{Gemma-2-9B}  &  HotpotQA  &  0.001  &  0.006  &  0.005 \\
  &  SQuAD 2.0  &  0.002  &  0.009  &  0.006 \\
  &  POTATO  &  0.004  &  0.021  &  0.014 \\
  &  BioASQ  &  0.004  &  0.021  &  0.012 \\
\hline
\multirow{4}{*}{Gemma-3-12B}  &  HotpotQA  &  0.000  &  0.001  &  0.001 \\
  &  SQuAD 2.0  &  0.000  &  0.002  &  0.002 \\
  &  POTATO  &  0.001  &  0.004  &  0.004 \\
  &  BioASQ  &  0.001  &  0.011  &  0.006 \\
\hline
\multirow{4}{*}{Llama-3.1-8B}  &  HotpotQA  &  0.009  &  0.025  &  0.018 \\
  &  SQuAD 2.0  &  0.010  &  0.029  &  0.019 \\
  &  POTATO  &  0.008  &  0.034  &  0.021 \\
  &  BioASQ  &  0.018  &  0.053  &  0.025 \\
\hline
\multirow{4}{*}{Mistral-v0.3-7B}  &  HotpotQA  &  0.001  &  0.009  &  0.007 \\
  &  SQuAD 2.0  &  0.002  &  0.015  &  0.010 \\
  &  POTATO  &  0.004  &  0.023  &  0.016 \\
  &  BioASQ  &  0.004  &  0.027  &  0.014 \\
\hline
\multirow{4}{*}{Phi-3.5-3.8B}  &  HotpotQA  &  0.003  &  0.013  &  0.010 \\
  &  SQuAD 2.0  &  0.002  &  0.013  &  0